In [ ]:
import pandas as pd
import re
from dateutil import parser as dateutil_parser

# ── Valores que indicam ausência de informação ─────────────────────────────────
INVALID_VALUES = {"?", "null", "NULL", "#N/D", "n/a", "N/A", "na", "NA", "none", "", "-", "--"}

def parse_date(raw) -> str | None:
    """
    Tenta interpretar qualquer string de data e retorna no formato dd/mm/yyyy.
    Retorna None se o valor for incoerente / ausente.
    """
    if pd.isna(raw):
        return None

    text = str(raw).strip()

    # Valores incoerentes
    if text.lower() in INVALID_VALUES:
        return None

    # Formato textual: "11 Jun 1984" ou "18 Jun 1956"
    month_abbr = r"(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)"
    if re.fullmatch(rf"\d{{1,2}} {month_abbr} \d{{4}}", text, re.IGNORECASE):
        try:
            dt = dateutil_parser.parse(text, dayfirst=True)
            return dt.strftime("%d/%m/%Y")
        except Exception:
            return None

    # Formatos numéricos — extrai os 3 componentes independente do separador
    parts = re.split(r"[-/.]", text)
    if len(parts) != 3:
        return None

    try:
        a, b, c = parts

        # Ano de 4 dígitos na PRIMEIRA posição → yyyy/mm/dd ou yyyy/dd/mm
        if len(a) == 4:
            year, p2, p3 = int(a), int(b), int(c)
            # p2 = mês, p3 = dia  (ex: 1985/6/11)
            month, day = (p2, p3) if 1 <= p2 <= 12 else (p3, p2)

        # Ano de 4 dígitos na ÚLTIMA posição → dd/mm/yyyy ou mm/dd/yyyy
        elif len(c) == 4:
            year, p1, p2 = int(c), int(a), int(b)
            # Se p1 > 12, só pode ser dd-mm
            if p1 > 12:
                day, month = p1, p2
            # Se p2 > 12, só pode ser mm-dd
            elif p2 > 12:
                month, day = p1, p2
            else:
                # Ambíguo: assume dd/mm (padrão brasileiro)
                day, month = p1, p2

        else:
            return None  # ano de 2 dígitos — não tratado

        # Valida a data montada
        from datetime import date
        dt = date(year, month, day)
        return dt.strftime("%d/%m/%Y")

    except (ValueError, TypeError):
        return None


# ── Exemplo de uso ─────────────────────────────────────────────────────────────
if __name__ == "__main__":

    # Substitua pelo caminho real do seu arquivo
    # df = pd.read_csv("sua_base.csv")

    # Dados de demonstração retirados das prints
    dados_exemplo = [
        "11 Jun 1984", "1985/6/11", "1990-06-10", "14 Jun 1975",
        "05 Jun 2008", "1975/6/14", "06-11-1986", "1959-06-18",
        "1968-06-15", "1972-06-14", "14 Jun 1973", "11/06/1987",
        "1971-06-15", "18 Jun 1956", "11/06/1985", "11 Jun 1987",
        "15 Jun 1968", "14 Jun 1975", "06-12-1980", "14/06/1975",
        "1970/6/15", "11/06/1984", "15/06/1970", "?",
        "06-08-1997", "06-10-1989", "NULL", "14/06/1974",
        "11/06/1985", "N/A", "1992/6/9", "06-14-1975",
    ]

    df = pd.DataFrame({"data_nascimento": dados_exemplo})

    df["data_nascimento_tratada"] = df["data_nascimento"].apply(parse_date)

    # Linhas com problemas (valor original era incoerente)
    problemas = df[df["data_nascimento_tratada"].isna()]

    print("=== Resultado ===")
    print(df.to_string(index=False))
    print(f"\n=== Linhas com valores incoerentes ({len(problemas)}) ===")
    print(problemas[["data_nascimento"]].to_string(index=False))

    # Para salvar:
    df = pd.read_csv("../bases/tabelas_brutas/cadastro_clientes.csv")
    df["data_nascimento"] = df["data_nascimento"].apply(parse_date)
    df.to_csv("../bases/tabelas_tratadas/cadastro_clientes_tratada.csv", index=False)

In [ ]:
import pandas as pd
import re
from dateutil import parser as dateutil_parser
from datetime import date

# ── Valores que indicam ausência de informação ─────────────────────────────────
INVALID_VALUES = {"?", "null", "NULL", "#N/D", "n/a", "N/A", "na", "NA", "none", "", "-", "--"}


def parse_date(raw) -> str | None:
    """
    Tenta interpretar qualquer string de data e retorna no formato dd/mm/yyyy.
    Retorna None se o valor for incoerente / ausente.
    """
    if pd.isna(raw):
        return None

    text = str(raw).strip()

    if text.lower() in INVALID_VALUES:
        return None

    # Formato textual: "11 Jun 1984"
    month_abbr = r"(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)"
    if re.fullmatch(rf"\d{{1,2}} {month_abbr} \d{{4}}", text, re.IGNORECASE):
        try:
            dt = dateutil_parser.parse(text, dayfirst=True)
            return dt.strftime("%d/%m/%Y")
        except Exception:
            return None

    # Formatos numéricos
    parts = re.split(r"[-/.]", text)
    if len(parts) != 3:
        return None

    try:
        a, b, c = parts

        if len(a) == 4:
            year, p2, p3 = int(a), int(b), int(c)
            month, day = (p2, p3) if 1 <= p2 <= 12 else (p3, p2)

        elif len(c) == 4:
            year, p1, p2 = int(c), int(a), int(b)
            if p1 > 12:
                day, month = p1, p2
            elif p2 > 12:
                month, day = p1, p2
            else:
                # Ambíguo: assume dd/mm (padrão brasileiro)
                day, month = p1, p2
        else:
            return None

        dt = date(year, month, day)
        return dt.strftime("%d/%m/%Y")

    except (ValueError, TypeError):
        return None


IDADE_MIN = 17
IDADE_MAX = 96


def parse_idade(raw) -> float | None:
    """Retorna a idade como float, ou None se o valor for incoerente/ausente."""
    if pd.isna(raw):
        return None
    text = str(raw).strip()
    if text.lower() in INVALID_VALUES:
        return None
    try:
        return float(text)
    except (ValueError, TypeError):
        return None


def idade_valida(val) -> bool:
    """Verifica se a idade está dentro do intervalo aceitável."""
    try:
        return IDADE_MIN <= float(val) <= IDADE_MAX
    except (TypeError, ValueError):
        return False


def calcular_idade(data_str: str) -> float | None:
    """Calcula a idade a partir de uma string dd/mm/yyyy já normalizada."""
    try:
        dt = dateutil_parser.parse(data_str, dayfirst=True).date()
        hoje = date.today()
        idade = hoje.year - dt.year - ((hoje.month, hoje.day) < (dt.month, dt.day))
        return float(idade)
    except Exception:
        return None


def tratar_base(df: pd.DataFrame) -> pd.DataFrame:
    """
    Recebe o DataFrame bruto e retorna ele tratado:
      - data_nascimento normalizada para dd/mm/yyyy
      - idade numérica, calculada a partir da data quando ausente ou fora do range
      - idades válidas: entre IDADE_MIN e IDADE_MAX anos
      - linhas onde AMBAS as colunas são incoerentes/inválidas são removidas
    """
    df = df.copy()

    # 1. Normaliza as duas colunas
    df["data_nascimento"] = df["data_nascimento"].apply(parse_date)
    df["idade"] = df["idade"].apply(parse_idade)

    # 2. Idade ausente ou fora do range → tenta calcular pela data de nascimento
    mask_corrigir = df["idade"].isna() | ~df["idade"].apply(
        lambda x: idade_valida(x) if pd.notna(x) else False
    )
    mask_tem_data = df["data_nascimento"].notna()

    mask_calcular = mask_corrigir & mask_tem_data
    df.loc[mask_calcular, "idade"] = df.loc[mask_calcular, "data_nascimento"].apply(calcular_idade)

    # 3. Remove linhas onde idade ainda está inválida/ausente E não há data de nascimento
    mask_deletar = ~df["idade"].apply(idade_valida) & df["data_nascimento"].isna()
    n_deletadas = mask_deletar.sum()
    df = df[~mask_deletar].reset_index(drop=True)

    print(f"Linhas removidas (sem dado aproveitável):     {n_deletadas}")
    print(f"Idades corrigidas via data_nascimento:        {mask_calcular.sum()}")
    return df


# ── Exemplo de uso ─────────────────────────────────────────────────────────────
if __name__ == "__main__":

    # Substitua pelo caminho real do seu arquivo:
    df = pd.read_csv("../bases/tabelas_tratadas/cadastro_clientes_tratada.csv")
    df = tratar_base(df)
    df.to_csv("../bases/tabelas_tratadas/cadastro_clientes_idades.csv", index=False)

    # Dados de demonstração baseados nas prints
    dados_exemplo = {
        "idade":           [ 34.0,        36.0,  41.0,         18.0,         -5.0,         999.0,        40.0,          0.0,          "?",          None,  None,         37.0,         32.0,         45.0,         29.0        ],
        "data_nascimento": ["14/06/1972",  None, "11/06/1985", "05/06/2008", "10/06/1988", "15/06/1971", "06/11/1986", "08/06/1996", "11/06/1986", "14/06/1975", None, "10/06/1989", "09/06/1994", "12/06/1981", "08/06/1997"],
    }

    df_bruto = pd.DataFrame(dados_exemplo)

    print("=== Base bruta ===")
    print(df_bruto.to_string(index=False))
    print()

    df_tratado = tratar_base(df_bruto)

    print()
    print("=== Base tratada ===")
    print(df_tratado.to_string(index=False))